![](https://media.licdn.com/dms/image/D5612AQEYszb2jxhegA/article-cover_image-shrink_720_1280/0/1686114807321?e=2147483647&v=beta&t=X6xQ60XNXz6Z7MogrTZh0dlrn-buWgYGWEZ8iX104cY)

# This is an IPL cricket SQLite database (2008–2016). 

*  It contains match-level data (teams, date, toss, winner, win margin/type, outcome, man of the match), player master data (name, batting hand, bowling skill), and player appearances per match (Player_Match).
*  It also includes ball-by-ball tables for deeper scoring/wicket analysis.


# Important Insights:

* Insight (1): The league format/volume expanded noticeably in seasons 4–6, then returned to a stable ~60 matches per season.
* Insight (2): Results are almost fully available, which makes the Match table strong for reliable KPI reporting.
* Insight (3): The dataset is dominated by standard completed matches; edge-case outcomes are rare.
* Insight (4): By Win_Type (important pattern):
    * One win type has margins tightly bounded 1–10 (avg ~6.32) → typically “close” finishes by definition.
    * Another win type can reach very large margins (avg ~30.32, max 144) → where “blowouts” happen.
* Insight (5): These are long-run “core” players with high participation across seasons.
* Insight (6): These players show high “match-level impact” frequency (MoM isn’t perfect, but it’s a strong proxy given table limits).
* Insight (7): Right-hand batters dominate, but left-handers are still a meaningful minority (tactically relevant for matchups).
* Insight (8): Bowling styles are diverse, but missing bowling skill values are non-trivial and should be handled carefully (Unknown category).
* Insight (9): Roles are heavily imbalanced (most entries are “Player”), so role-based comparisons should account for sample-size bias.


In [ ]:
import numpy as np
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

# connecting to database

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = "/kaggle/input/ipldatabase/database.sqlite"
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query("""
SELECT name, type
FROM sqlite_master
WHERE type IN ('table','view')
ORDER BY type, name;
""", conn)

tables

In [ ]:
query = """

SELECT *
FROM sqlite_master
WHERE TYPE="table";

"""
tables = pd.read_sql(query,conn)

tables['name']

In [ ]:
player = pd.read_sql("SELECT * FROM Player", conn)
player

In [ ]:
Match = pd.read_sql("SELECT * FROM Match", conn)
Match

In [ ]:
Player_Match = pd.read_sql("SELECT * FROM Player_Match", conn)
Player_Match

In [ ]:
player.isna().sum()

In [ ]:
Match.isna().sum()

In [ ]:
Player_Match.isna().sum()

In [ ]:
player.info()

In [ ]:
player['Bowling_skill'] = player['Bowling_skill'].interpolate()

In [ ]:
player.isna().sum()

In [ ]:
Match.info()

In [ ]:
Match['Match_Winner'] = Match['Match_Winner'].interpolate()

In [ ]:
Match['Man_of_the_Match'] = Match['Man_of_the_Match'].interpolate()

In [ ]:
Match.isna().sum()

In [ ]:
Match['Win_Margin'] = Match['Win_Margin'].interpolate()

In [ ]:
Match

In [ ]:
Match['Match_Date'] = pd.to_datetime(Match['Match_Date'])

In [ ]:
Match.info()

In [ ]:
Match

In [ ]:
Match['Match_Winner'].nunique()

In [ ]:
Match['Match_Winner'].unique()

In [ ]:
Win_By = pd.read_sql("SELECT * FROM Win_By", conn)
Win_By

# Analysis + Visulization

# How did match volume change across seasons (Season_Id)?

In [ ]:
matches_by_season = pd.read_sql_query("""
SELECT Season_Id, COUNT(*) AS Matches
FROM Match
GROUP BY Season_Id
ORDER BY Matches DESC;
""", conn)
matches_by_season

In [ ]:
plt.figure()
plt.plot(matches_by_season["Season_Id"], matches_by_season["Matches"], marker="o")
plt.xlabel("Season_Id")
plt.ylabel("Matches")
plt.title("Match Volume Across Seasons")
plt.show()

# Distribution of match outcomes (Outcome_type IDs)

In [ ]:
Outcome = pd.read_sql("SELECT * FROM Outcome",conn)
Outcome

In [ ]:
outcome_dist = pd.read_sql_query("""
SELECT m.Outcome_type AS Outcome_Id,o.Outcome_Type , COUNT(*) AS Matches
FROM Match m
JOIN Outcome o
on m.Outcome_type = o.Outcome_Id
GROUP BY Outcome_Id,o.Outcome_Type
ORDER BY Outcome_Id;
""", conn)
outcome_dist

In [ ]:
plt.figure()
plt.bar(outcome_dist["Outcome_Type"], outcome_dist["Matches"])
plt.xlabel("Outcome Type (ID)")
plt.ylabel("Matches")
plt.title("Distribution of Match Outcomes (Outcome Type IDs)")
plt.show()

# Are matches typically close or one-sided (using Win_Margin)?

In [ ]:
df = pd.read_sql_query("""
SELECT m.Win_Type AS Win_Id,w.Win_Type , Win_Margin
FROM Match m
JOIN Win_By w
on m.Win_Type = w.Win_Id
WHERE Match_Winner IS NOT NULL AND Win_Margin IS NOT NULL AND Win_Margin >= 0
""", conn)
df

In [ ]:
px.histogram(df, x="Win_Margin", nbins=30, facet_col="Win_Type",
             title="Win_Margin by Win_Type").show()

In [ ]:
player

In [ ]:
player.info()

In [ ]:
player['DOB'] = pd.to_datetime(player['DOB'])

# Who are the most “available” players (most matches played)?

In [ ]:
matches_played = pd.read_sql_query("""
SELECT p.Player_Name, COUNT(DISTINCT pm.Match_Id) AS Matches_Played
FROM Player_Match pm
JOIN Player p ON p.Player_Id = pm.Player_Id
GROUP BY p.Player_Id
ORDER BY Matches_Played DESC
LIMIT 20;
""", conn)
matches_played

In [ ]:
px.bar(matches_played, x="Player_Name", y="Matches_Played",
       title="Top 20 Players by Matches Played (Availability)",) \
  .update_layout(xaxis_title="", yaxis_title="Matches Played") \
  .show()

# Who has the most Man of the Match awards?

In [ ]:
mom_totals = pd.read_sql_query("""
SELECT p.Player_Name, COUNT(*) AS MoM_Awards
FROM Match m
JOIN Player p ON p.Player_Id = m.Man_of_the_Match
WHERE m.Man_of_the_Match IS NOT NULL
GROUP BY p.Player_Id
ORDER BY MoM_Awards DESC
LIMIT 20;
""", conn)
mom_totals

In [ ]:
px.bar(mom_totals, x="Player_Name", y="MoM_Awards",
       title="Top 20 Players by Man of the Match Awards") \
  .update_layout(xaxis_title="", yaxis_title="MoM Awards") \
  .show()

In [ ]:
player['Bowling_skill'].unique()

# Batting hand distribution

In [ ]:
Batting_Style = pd.read_sql("SELECT * FROM Batting_Style",conn)
Batting_Style

In [ ]:
bat = pd.read_sql_query("""
SELECT p.Batting_hand AS Batting_Id,bs.Batting_hand, COUNT(*) AS Players
FROM Player p
JOIN Batting_Style bs
on p.Batting_hand = bs.Batting_Id
GROUP BY Batting_Id
ORDER BY Players DESC;
""", conn)
bat

In [ ]:
fig = px.pie(bat, names="Batting_hand", values="Players",
       title="Player Distribution by Batting Hand")
fig.update_traces(textinfo='percent+label',pull=[0.05])
fig.update_layout(width=900, height=600)
fig.show()

# Bowling skill distribution (including NULL)

In [ ]:
Bowling_Style = pd.read_sql("SELECT * FROM Bowling_Style",conn)
Bowling_Style

In [ ]:
bowl = pd.read_sql_query("""
SELECT COALESCE(CAST(p.Bowling_skill AS TEXT), 'NULL/Unknown') AS Bowling_Id,bs.Bowling_skill,
       COUNT(*) AS Players
FROM Player p
JOIN Bowling_Style bs
on p.Bowling_skill = bs.Bowling_Id
GROUP BY COALESCE(CAST(p.Bowling_skill AS TEXT), 'NULL/Unknown')
ORDER BY Players DESC;
""", conn)
bowl

In [ ]:
fig1 = px.pie(bowl, names="Bowling_skill", values="Players",
       title="Player Distribution by Bowling Skill (Bowling_skill)")
fig1.update_traces(textinfo='percent+label',pull=[0.05])
fig1.update_layout(width=900, height=600)
fig1.show()

In [ ]:
Player_Match

In [ ]:
Rolee = pd.read_sql("SELECT * FROM Rolee", conn)
Rolee

# How imbalanced are player roles (Role_Id) in appearances?

In [ ]:
role_dist = pd.read_sql_query("""
SELECT pm.Role_Id,r.Role_Desc, COUNT(*) AS Appearances
FROM Player_Match pm
JOIN Rolee r
on pm.Role_Id = r.Role_Id
GROUP BY pm.Role_Id
ORDER BY Appearances DESC;
""", conn)
role_dist

In [ ]:
px.bar(role_dist, x="Role_Desc", y="Appearances",
       title="Role Imbalance (Role Id vs Appearances)").show()

![](https://media.istockphoto.com/id/1087068566/video/thank-you-animation.jpg?s=640x640&k=20&c=MtDyJsTw32NdZLNxwLk2BmWYH1L7n9oUgp46qXabOfE=)